In [8]:
!pip install -q llama-index

In [17]:
import sys
import time
import pickle
import logging
from pathlib import Path
from typing import List, Optional, Tuple, Union
import numpy as np
from scipy.sparse import csr_matrix, save_npz, load_npz
from llama_index.core import Document
# Add parent directory to Python path
# Assuming your project root is `/work`, add it to the system path
project_root = Path('/work')
sys.path.append(str(project_root))
from src.utils.data_utils import load_documents, select_random_documents
from src.utils.evaluation import (
    perform_analysis,
    perform_single_match_analysis,
    print_single_match_analysis,
    print_sample_results,
    calculate_prefix_success_at_k,
    perform_prefix_match_analysis,
    print_prefix_match_analysis
)
from contextlib import contextmanager

# ---------------------------
# Timing context manager
# ---------------------------
@contextmanager
def timer(name):
    omitted_names = {
        "Selecting a random document for sanity check",
        "Encoding query",
        "Searching in Milvus",
        "Processing search results"
    }
    start = time.time()
    yield
    end = time.time()
    if name not in omitted_names and not name.startswith("Retrieval for document"):
        print(f"[Timing] {name}: {end - start:.4f} seconds")

logging.basicConfig(level=logging.INFO)

def main(index_path_texto, index_path_resumen, file_identifier, batch_size=500, top_k=10):
    start_total = time.time()

    # Load persisted documents
    with open(index_path_texto / "documents.pkl", "rb") as f:
        documents_texto = pickle.load(f)
    with open(index_path_resumen / "documents.pkl", "rb") as f:
        documents_resumen = pickle.load(f)

    # Load persisted vectorizers and embeddings
    with open(index_path_texto / "vectorizer.pkl", "rb") as f:
        vectorizer_texto = pickle.load(f)
    embeddings_texto = load_npz(index_path_texto / "embeddings.npz")

    with open(index_path_resumen / "vectorizer.pkl", "rb") as f:
        vectorizer_resumen = pickle.load(f)
    # (We won’t use vectorizer_resumen here since we’re reusing vectorizer_texto for retrieval.)

    # Build a mapping from texto item keys to their indices for fast lookup
    texto_mapping = {doc.metadata.get("item_key"): idx for idx, doc in enumerate(documents_texto)}

    # Select a random sample from the resumen documents
    sample_size = min(len(documents_resumen), 16590)
    random_resumen = select_random_documents(documents_resumen, sample_size)
    resumen_keys, resumen_texts = zip(*random_resumen)  # unzip into separate lists

    all_results = []
    total_batches = (sample_size + batch_size - 1) // batch_size
    start_batches = time.time()

    # Batch processing: transform and compute scores in batches
    for batch_num, start_idx in enumerate(range(0, sample_size, batch_size), 1):
        end_idx = min(start_idx + batch_size, sample_size)
        batch_keys = resumen_keys[start_idx:end_idx]
        batch_texts = resumen_texts[start_idx:end_idx]

        # Batch-transform resumen texts using vectorizer_texto
        batch_query_embeddings = vectorizer_texto.transform(batch_texts)
        # Compute similarity scores in one sparse matrix multiplication
        # scores_batch: shape (batch_size, number of texto documents)
        scores_batch = (batch_query_embeddings @ embeddings_texto.T).toarray()

        # Process each query in the batch
        for i, score_row in enumerate(scores_batch):
            overall_idx = start_idx + i
            item_key = batch_keys[i]
            resumen_text = batch_texts[i]

            # Extract top_k indices and sort them by score descending
            topk_idx_unsorted = np.argpartition(score_row, -top_k)[-top_k:]
            topk_idx = topk_idx_unsorted[np.argsort(score_row[topk_idx_unsorted])[::-1]]

            # Build the list of top_k results
            texto_results = [{
                "item_key": documents_texto[idx].metadata.get("item_key", f"Doc-{idx}"),
                "text": documents_texto[idx].text,
                "score": score_row[idx]
            } for idx in topk_idx]

            # Get target score from matching texto document (if exists)
            texto_index = texto_mapping.get(item_key, None)
            if texto_index is not None:
                target_score = score_row[texto_index]
                target_text = documents_texto[texto_index].text
            else:
                target_score = 0.0
                target_text = "No matching texto found"

            # Append results for this resumen document
            all_results.append({
                'resumen_item_key': item_key,
                'resumen_text': resumen_text,
                'target_text': target_text,
                'target_score': target_score,
                'texto_results': texto_results
            })

        # Logging progress for each batch
        elapsed_batches = time.time() - start_batches
        batches_remaining = total_batches - batch_num
        avg_batch_time = elapsed_batches / batch_num
        estimated_remaining_sec = batches_remaining * avg_batch_time
        est_min, est_sec = divmod(estimated_remaining_sec, 60)
        logging.info(
            f"Batch {batch_num}/{total_batches} processed. "
            f"Estimated time to completion: {int(est_min)}m {int(est_sec)}s"
        )

    # Perform overall analysis
    analysis_results = perform_analysis(all_results)
    print("\nAnalysis Results:")
    for metric, value in analysis_results.items():
        print(f"{metric}: {value:.4f}")

    # Perform single-match analysis and print results
    metrics = perform_single_match_analysis(all_results, top_k=top_k)
    print_single_match_analysis(metrics, top_k=top_k)

    # Perform prefix-matching analysis
    with timer("Performing prefix-matching analysis"):
        prefix_length = 6  # Match first 6 characters of item_key
        prefix_match_metrics = perform_prefix_match_analysis(
            all_results, 
            top_k=10, 
            prefix_length=prefix_length
        )
        print_prefix_match_analysis(
            prefix_match_metrics, 
            top_k=10, 
            prefix_length=prefix_length
        )

    # Print some sample retrieval results
    print_sample_results(all_results)

    total_elapsed = time.time() - start_total
    tot_min, tot_sec = divmod(total_elapsed, 60)
    print(f"\nTotal execution time: {int(tot_min)}m {int(tot_sec)}s")


In [19]:
import pickle
import numpy as np
from scipy.sparse import load_npz
from pathlib import Path
import pandas as pd

def inspect_tfidf_index(index_path, num_documents=5, num_features=20):
    """
    Inspect a TF-IDF index to understand its structure and content
    """
    print(f"Inspecting TF-IDF index at: {index_path}")
    print("=" * 60)
    
    # Load the components
    with open(index_path / "vectorizer.pkl", "rb") as f:
        vectorizer = pickle.load(f)
    
    embeddings = load_npz(index_path / "embeddings.npz")
    
    with open(index_path / "documents.pkl", "rb") as f:
        documents = pickle.load(f)
    
    # Basic statistics
    print(f"📊 BASIC STATISTICS")
    print(f"Number of documents: {len(documents)}")
    print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
    print(f"Embeddings shape: {embeddings.shape}")
    print(f"Matrix density: {embeddings.nnz / (embeddings.shape[0] * embeddings.shape[1]):.6f}")
    print(f"Total non-zero entries: {embeddings.nnz:,}")
    print()
    
    # TF-IDF specific information
    print(f"🔢 TF-IDF PARAMETERS")
    print(f"Normalization: {vectorizer.norm}")
    print(f"Use IDF: {vectorizer.use_idf}")
    print(f"Smooth IDF: {vectorizer.smooth_idf}")
    print(f"Sublinear TF: {vectorizer.sublinear_tf}")
    print()
    
    # IDF values inspection
    print(f"📉 IDF VALUES ANALYSIS")
    feature_names = vectorizer.get_feature_names_out()
    idf_values = vectorizer.idf_
    
    print(f"IDF statistics:")
    print(f"  Min IDF: {idf_values.min():.4f}")
    print(f"  Max IDF: {idf_values.max():.4f}")
    print(f"  Mean IDF: {idf_values.mean():.4f}")
    print(f"  Std IDF: {idf_values.std():.4f}")
    
    # Terms with highest IDF (rarest terms)
    highest_idf_indices = idf_values.argsort()[-10:][::-1]
    print(f"\nTerms with highest IDF (rarest):")
    for i, idx in enumerate(highest_idf_indices, 1):
        term = feature_names[idx]
        idf = idf_values[idx]
        print(f"  {i:2d}. '{term}': {idf:.4f}")
    
    # Terms with lowest IDF (most common)
    lowest_idf_indices = idf_values.argsort()[:10]
    print(f"\nTerms with lowest IDF (most common):")
    for i, idx in enumerate(lowest_idf_indices, 1):
        term = feature_names[idx]
        idf = idf_values[idx]
        print(f"  {i:2d}. '{term}': {idf:.4f}")
    print()
    
    # Vocabulary inspection
    print(f"🔤 VOCABULARY SAMPLE")
    vocab_items = list(vectorizer.vocabulary_.items())
    vocab_sample = sorted(vocab_items)[:num_features]  # First N alphabetically
    for word, idx in vocab_sample:
        idf_val = idf_values[idx]
        print(f"  '{word}' -> index {idx}, IDF: {idf_val:.4f}")
    print(f"  ... and {len(vocab_items) - num_features} more terms")
    print()
    
    # Document-level inspection
    print(f"📄 DOCUMENT SAMPLES")
    for i in range(min(num_documents, len(documents))):
        doc = documents[i]
        doc_text = doc.text if hasattr(doc, 'text') else str(doc)
        
        print(f"\nDocument {i+1}:")
        print(f"  Preview: {doc_text[:100]}...")
        
        # Get TF-IDF representation for this document
        doc_vector = embeddings[i].toarray().flatten()
        non_zero_indices = np.nonzero(doc_vector)[0]
        
        print(f"  Non-zero terms: {len(non_zero_indices)}")
        print(f"  Vector L2 norm: {np.linalg.norm(doc_vector):.4f}")
        print(f"  Max TF-IDF score: {doc_vector.max():.4f}")
        print(f"  Mean TF-IDF score (non-zero): {doc_vector[non_zero_indices].mean():.4f}")
        
        # Show top TF-IDF terms in this document
        if len(non_zero_indices) > 0:
            doc_term_scores = [(feature_names[idx], doc_vector[idx]) for idx in non_zero_indices]
            doc_term_scores.sort(key=lambda x: x[1], reverse=True)
            
            print(f"  Top TF-IDF terms in this document:")
            for term, score in doc_term_scores[:10]:  # Top 10 terms
                idf_val = idf_values[vectorizer.vocabulary_[term]]
                print(f"    '{term}': {score:.4f} (IDF: {idf_val:.4f})")
    print()

def compare_tfidf_document_vectors(index_path, doc_indices=[0, 1]):
    """
    Compare TF-IDF vectors between specific documents
    """
    print(f"🔍 COMPARING TF-IDF DOCUMENTS")
    print("=" * 40)
    
    # Load components
    with open(index_path / "vectorizer.pkl", "rb") as f:
        vectorizer = pickle.load(f)
    
    embeddings = load_npz(index_path / "embeddings.npz")
    feature_names = vectorizer.get_feature_names_out()
    idf_values = vectorizer.idf_
    
    with open(index_path / "documents.pkl", "rb") as f:
        documents = pickle.load(f)
    
    for doc_idx in doc_indices:
        if doc_idx >= len(documents):
            print(f"Document {doc_idx} doesn't exist (only {len(documents)} documents)")
            continue
            
        doc = documents[doc_idx]
        doc_text = doc.text if hasattr(doc, 'text') else str(doc)
        doc_vector = embeddings[doc_idx].toarray().flatten()
        
        print(f"\nDocument {doc_idx}:")
        print(f"  Text preview: {doc_text[:150]}...")
        print(f"  Vector L2 norm: {np.linalg.norm(doc_vector):.4f}")
        print(f"  Non-zero terms: {len(np.nonzero(doc_vector)[0])}")
        print(f"  Max TF-IDF score: {doc_vector.max():.4f}")
        
        # Show non-zero entries with TF-IDF scores and IDF values
        non_zero_indices = np.nonzero(doc_vector)[0]
        if len(non_zero_indices) > 0:
            terms_and_scores = [(feature_names[idx], doc_vector[idx], idf_values[idx]) 
                               for idx in non_zero_indices]
            terms_and_scores.sort(key=lambda x: x[1], reverse=True)
            
            print(f"  TF-IDF representation (top 15 terms):")
            print(f"    {'Term':<15} {'TF-IDF':<8} {'IDF':<8}")
            print(f"    {'-'*15} {'-'*8} {'-'*8}")
            for term, tfidf_score, idf_val in terms_and_scores[:15]:
                print(f"    {term:<15} {tfidf_score:<8.4f} {idf_val:<8.4f}")

def show_tfidf_as_dataframe(index_path, max_docs=5, max_features=20):
    """
    Show TF-IDF as a DataFrame for easy viewing
    """
    print(f"📋 TF-IDF AS TABLE")
    print("=" * 40)
    
    # Load components
    with open(index_path / "vectorizer.pkl", "rb") as f:
        vectorizer = pickle.load(f)
    
    embeddings = load_npz(index_path / "embeddings.npz")
    feature_names = vectorizer.get_feature_names_out()
    idf_values = vectorizer.idf_
    
    # Get terms with highest max TF-IDF scores across all documents
    max_tfidf_per_term = np.array(embeddings.max(axis=0)).flatten()
    top_indices = max_tfidf_per_term.argsort()[-max_features:][::-1]
    top_features = [feature_names[idx] for idx in top_indices]
    
    # Create DataFrame
    dense_matrix = embeddings[:max_docs, top_indices].toarray()
    df = pd.DataFrame(dense_matrix, 
                     columns=top_features,
                     index=[f"Doc_{i}" for i in range(max_docs)])
    
    print(f"TF-IDF matrix (first {max_docs} docs, top {max_features} terms by max score):")
    print(df.round(4))
    
    # Show IDF values for these terms
    idf_df = pd.DataFrame({
        'Term': top_features,
        'IDF': [idf_values[vectorizer.vocabulary_[term]] for term in top_features],
        'Max_TFIDF': [max_tfidf_per_term[vectorizer.vocabulary_[term]] for term in top_features]
    })
    print(f"\nIDF values for displayed terms:")
    print(idf_df.round(4))
    
    return df, idf_df

def analyze_tfidf_distribution(index_path):
    """
    Analyze the distribution of TF-IDF scores
    """
    print(f"📈 TF-IDF SCORE DISTRIBUTION")
    print("=" * 40)
    
    # Load components
    embeddings = load_npz(index_path / "embeddings.npz")
    
    # Get all non-zero TF-IDF scores
    all_scores = embeddings.data
    
    print(f"TF-IDF score statistics:")
    print(f"  Total non-zero scores: {len(all_scores):,}")
    print(f"  Min score: {all_scores.min():.6f}")
    print(f"  Max score: {all_scores.max():.6f}")
    print(f"  Mean score: {all_scores.mean():.6f}")
    print(f"  Median score: {np.median(all_scores):.6f}")
    print(f"  Std deviation: {all_scores.std():.6f}")
    
    # Score percentiles
    percentiles = [90, 95, 99, 99.9]
    print(f"\nScore percentiles:")
    for p in percentiles:
        score = np.percentile(all_scores, p)
        print(f"  {p:4.1f}th percentile: {score:.6f}")

In [18]:
index_path_texto = Path(f'/work/sandbox/tf_idf/indexes/texto')
index_path_resumen = Path(f'/work/sandbox/tf_idf/indexes/resumen')
main(index_path_texto, index_path_resumen, 'OEB')

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 33s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 32s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 30s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 29s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 27s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 26s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.8051
Precision: 0.0805
F1 Score: 0.1464
Mean Reciprocal Rank (MRR): 0.6759
Mean Average Precision (MAP): 0.6759

Single-Match Analysis Results:
Success@10             : 80.51%
MRR: 0.6759
Mean Position: 1.81
Median Position: 1.0
Perfect Matches: 62.59%

Position Distribution:
------------------------------
Position_1: 62.59%
Position_2: 4.22%
Position_3: 3.10%
Position_4: 1.37%
Position_5: 3.80%
Position_6: 2.24%
Position_7: 0.80%
Position_8: 0.33%
Position_9: 1.55%
Position_10: 0.51%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 98.24%
Prefix_MRR: 0.9561
Prefix_Mean_Position: 1.14
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 94.74%

Prefix Position Distribution:
----------------------------------------
Position 1    : 94.74%
Position 2    : 0.26%
Position 3    : 1.15%
Position 4    : 0.14%
Position 5    : 0.96%
Position 6    : 0.08%
Position 7    : 0.37%
Position 8    : 0.04%
Position 9    : 0.48%
Position 1

In [20]:
index_path = Path('/work/sandbox/tf_idf/indexes/texto')
inspect_tfidf_index(index_path, num_documents=5, num_features=20)

Inspecting TF-IDF index at: /work/sandbox/tf_idf/indexes/texto
📊 BASIC STATISTICS
Number of documents: 47514
Vocabulary size: 326
Embeddings shape: (47514, 326)
Matrix density: 0.173278
Total non-zero entries: 2,684,003

🔢 TF-IDF PARAMETERS
Normalization: l2
Use IDF: True
Smooth IDF: True
Sublinear TF: False

📉 IDF VALUES ANALYSIS
IDF statistics:
  Min IDF: 1.0000
  Max IDF: 11.0757
  Mean IDF: 5.2399
  Std IDF: 2.8530

Terms with highest IDF (rarest):
   1. 'zanjas': 11.0757
   2. '5t': 11.0757
   3. 'presión': 11.0757
   4. 'presion': 11.0757
   5. 'conexión': 11.0757
   6. 'polipropileno': 11.0757
   7. 'placa': 11.0757
   8. '29mm': 10.3825
   9. '36mm': 10.3825
  10. '21mm': 10.3825

Terms with lowest IDF (most common):
   1. 'de': 1.0000
   2. 'ejecución': 1.0006
   3. 'condiciones': 1.0006
   4. 'mantenimiento': 1.0013
   5. 'trabajo': 1.0013
   6. 'banda': 1.0013
   7. 'y': 1.0048
   8. 'la': 1.0187
   9. 'canalización': 1.0298
  10. 'tubos': 1.0381

🔤 VOCABULARY SAMPLE
  '0' -

In [21]:
compare_tfidf_document_vectors(index_path, doc_indices=[0, 1, 2])

🔍 COMPARING TF-IDF DOCUMENTS

Document 0:
  Text preview: ZANJAS, CANALIZACIONES Y TUBOS...
  Vector L2 norm: 1.0000
  Non-zero terms: 4
  Max TF-IDF score: 0.8453
  TF-IDF representation (top 15 terms):
    Term            TF-IDF   IDF     
    --------------- -------- --------
    zanjas          0.8453   11.0757 
    canalizaciones  0.5227   6.8488  
    tubos           0.0792   1.0381  
    y               0.0767   1.0048  

Document 1:
  Text preview: Ejecución de canalización para línea subterránea de doble circuito de 220 o 400 kV de cables entubados y tubos embebidos en hormigón HM-20, de dimensi...
  Vector L2 norm: 1.0000
  Non-zero terms: 110
  Max TF-IDF score: 0.3788
  TF-IDF representation (top 15 terms):
    Term            TF-IDF   IDF     
    --------------- -------- --------
    de              0.3788   1.0000  
    exterior        0.2648   8.1579  
    para            0.2225   3.4269  
    250             0.1910   8.8244  
    doble           0.1910   8.8244  
    1

In [22]:
df, idf_df = show_tfidf_as_dataframe(index_path, max_docs=5, max_features=25)

📋 TF-IDF AS TABLE
TF-IDF matrix (first 5 docs, top 25 terms by max score):
         0
Doc_0  0.0
Doc_1  0.0
Doc_2  0.0
Doc_3  0.0
Doc_4  0.0

IDF values for displayed terms:
  Term     IDF                                          Max_TFIDF
0    0  4.7832    (0, 0)\t0.2905724201149618\n  (0, 1)\t0.1753...


In [23]:
analyze_tfidf_distribution(index_path)

📈 TF-IDF SCORE DISTRIBUTION
TF-IDF score statistics:
  Total non-zero scores: 2,684,003
  Min score: 0.010767
  Max score: 0.847971
  Mean score: 0.097139
  Median score: 0.060374
  Std deviation: 0.090922

Score percentiles:
  90.0th percentile: 0.168024
  95.0th percentile: 0.215934
  99.0th percentile: 0.659582
  99.9th percentile: 0.716440


In [24]:
index_path_texto_restricted = Path(f'/work/sandbox/tf_idf/indexes/texto_restricted')
index_path_resumen_restricted = Path(f'/work/sandbox/tf_idf/indexes/resumen_restricted')
main(index_path_texto_restricted, index_path_resumen_restricted, 'OEB')

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 17s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 16s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 16s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 15s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 15s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 14s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 13s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 13s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 12s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.6138
Precision: 0.0614
F1 Score: 0.1116
Mean Reciprocal Rank (MRR): 0.2407
Mean Average Precision (MAP): 0.2407

Single-Match Analysis Results:
Success@10             : 61.38%
MRR: 0.2407
Mean Position: 4.35
Median Position: 4.0
Perfect Matches: 10.67%

Position Distribution:
------------------------------
Position_1: 10.67%
Position_2: 10.69%
Position_3: 7.87%
Position_4: 7.75%
Position_5: 4.55%
Position_6: 4.62%
Position_7: 3.63%
Position_8: 3.76%
Position_9: 3.99%
Position_10: 3.85%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 94.85%
Prefix_MRR: 0.5960
Prefix_Mean_Position: 2.83
Prefix_Median_Position: 2.0
Prefix_Perfect_Matches: 45.65%

Prefix Position Distribution:
----------------------------------------
Position 1    : 45.65%
Position 2    : 11.84%
Position 3    : 8.44%
Position 4    : 7.16%
Position 5    : 6.23%
Position 6    : 5.26%
Position 7    : 4.32%
Position 8    : 2.80%
Position 9    : 1.96%
Position

In [25]:
inspect_tfidf_index(index_path_texto_restricted, num_documents=5, num_features=20)

Inspecting TF-IDF index at: /work/sandbox/tf_idf/indexes/texto_restricted
📊 BASIC STATISTICS
Number of documents: 47514
Vocabulary size: 153
Embeddings shape: (47514, 153)
Matrix density: 0.141064
Total non-zero entries: 1,025,486

🔢 TF-IDF PARAMETERS
Normalization: l2
Use IDF: True
Smooth IDF: True
Sublinear TF: False

📉 IDF VALUES ANALYSIS
IDF statistics:
  Min IDF: 1.0006
  Max IDF: 11.7688
  Mean IDF: 7.4015
  Std IDF: 4.0211

Terms with highest IDF (rarest):
   1. 'cualquier franja horaria': 11.7688
   2. 'no aplica': 11.7688
   3. '3 <= i < 5 horas': 11.7688
   4. 'acción': 11.7688
   5. '3"': 11.7688
   6. '90mm': 11.7688
   7. 'bajo vías': 11.7688
   8. 'nº': 11.7688
   9. 'normal': 11.7688
  10. 'nocturno excepcional': 11.7688

Terms with lowest IDF (most common):
   1. 'condiciones': 1.0006
   2. 'ejecución': 1.0006
   3. 'banda': 1.0013
   4. 'trabajo': 1.0013
   5. 'mantenimiento': 1.0013
   6. 'canalización': 1.0298
   7. 'tubos': 1.0381
   8. 'diámetro': 1.0428
   9. 'rel

In [27]:
compare_tfidf_document_vectors(index_path_texto_restricted, doc_indices=[0, 1, 2])

🔍 COMPARING TF-IDF DOCUMENTS

Document 0:
  Text preview: ZANJAS, CANALIZACIONES Y TUBOS...
  Vector L2 norm: 1.0000
  Non-zero terms: 3
  Max TF-IDF score: 0.8478
  TF-IDF representation (top 15 terms):
    Term            TF-IDF   IDF     
    --------------- -------- --------
    zanjas          0.8478   11.0757 
    canalizaciones  0.5243   6.8488  
    tubos           0.0795   1.0381  

Document 1:
  Text preview: Ejecución de canalización para línea subterránea de doble circuito de 220 o 400 kV de cables entubados y tubos embebidos en hormigón HM-20, de dimensi...
  Vector L2 norm: 1.0000
  Non-zero terms: 33
  Max TF-IDF score: 0.4112
  TF-IDF representation (top 15 terms):
    Term            TF-IDF   IDF     
    --------------- -------- --------
    doble           0.4112   8.8244  
    m               0.3320   4.7504  
    2               0.2949   3.1645  
    mm              0.2309   1.1010  
    profundidad     0.2214   4.7504  
    material        0.2204   4.7291  
    ca

In [26]:
df, idf_df = show_tfidf_as_dataframe(index_path_texto_restricted, max_docs=5, max_features=25)

📋 TF-IDF AS TABLE
TF-IDF matrix (first 5 docs, top 25 terms by max score):
         0
Doc_0  0.0
Doc_1  0.0
Doc_2  0.0
Doc_3  0.0
Doc_4  0.0

IDF values for displayed terms:
  Term     IDF                                          Max_TFIDF
0    0  4.7832    (0, 0)\t0.5439387975762077\n  (0, 2)\t0.5596...


In [31]:
compare_tfidf_document_vectors(index_path_resumen_restricted, doc_indices=[0, 1, 2])

🔍 COMPARING TF-IDF DOCUMENTS

Document 0:
  Text preview: ZANJAS, CANALIZACIONES Y TUBOS...
  Vector L2 norm: 1.0000
  Non-zero terms: 3
  Max TF-IDF score: 0.6878
  TF-IDF representation (top 15 terms):
    Term            TF-IDF   IDF     
    --------------- -------- --------
    canalizaciones  0.6878   11.0757 
    zanjas          0.6878   11.0757 
    tubos           0.2320   3.7361  

Document 1:
  Text preview: Ejecución de canalización para línea subterranea doble circuito de 220 ó 400 kV en terreno  blando ,  sin reposición  de pavimento. (-/-/Volumen relev...
  Vector L2 norm: 1.0000
  Non-zero terms: 12
  Max TF-IDF score: 0.3506
  TF-IDF representation (top 15 terms):
    Term            TF-IDF   IDF     
    --------------- -------- --------
    blando          0.3506   9.2039  
    220             0.3108   8.1579  
    400             0.3108   8.1579  
    circuito        0.3108   8.1579  
    doble           0.3108   8.1579  
    kv              0.3108   8.1579  
    lí

In [28]:
def load_vocabulary(vocab_path):
    """Load vocabulary dictionary from pickle file"""
    try:
        with open(vocab_path, 'rb') as f:
            vocab_dict = pickle.load(f)
        print(f"Loaded vocabulary with {len(vocab_dict)} terms from {vocab_path}")
        return vocab_dict
    except Exception as e:
        print(f"Error loading vocabulary from {vocab_path}: {e}")
        return None

In [30]:
vocab_file = Path(f'/work/sandbox/tf_idf/vocab_dict.pkl')
vocab_dict = load_vocabulary(vocab_file)
vocab_dict

Loaded vocabulary with 153 terms from /work/sandbox/tf_idf/vocab_dict.pkl


{'0': 0,
 '0,80 a 1,00 m': 1,
 '1': 2,
 '1 o 2': 3,
 "1''": 4,
 '1,10 m': 5,
 '110': 6,
 '110mm': 7,
 '12': 8,
 '16': 9,
 '160mm': 10,
 '18': 11,
 '2': 12,
 '2"': 13,
 "2''": 14,
 '20': 15,
 '200mm': 16,
 '21mm': 17,
 '220': 18,
 '24': 19,
 '29mm': 20,
 '3': 21,
 "3 1/2''": 22,
 '3 <= i < 5 horas': 23,
 '3"': 24,
 "3''": 25,
 '36mm': 26,
 '4': 27,
 "4''": 28,
 '400': 29,
 '40mm': 30,
 '48mm': 31,
 '5': 32,
 "5''": 33,
 '50mm': 34,
 '5t': 35,
 '6': 36,
 "6''": 37,
 '60': 38,
 '8': 39,
 '90mm': 40,
 'acción': 41,
 'acero': 42,
 'adosada': 43,
 'alma': 44,
 'anchura': 45,
 'andén': 46,
 'arqueta': 47,
 'bajo vías': 48,
 'balasto': 49,
 'banda': 50,
 'blando': 51,
 'cable': 52,
 'cables': 53,
 'canaleta': 54,
 'canalizaciones': 55,
 'canalización': 56,
 'cata': 57,
 'circuito': 58,
 'cms': 59,
 'con reposición': 60,
 'con topo': 61,
 'condiciones': 62,
 'conexión': 63,
 'corrugado': 64,
 'cruce': 65,
 'cruce de carretera': 66,
 'cualquier condición de ejecución': 67,
 'cualquier franja hor